In [1]:
import warnings
import pandas as pd
import numpy as np
import os
import glob

warnings.filterwarnings("ignore", category =UserWarning, module="openpyxl")
tranaction_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Payment\Tranaction_Payment\tranctions.csv"
tranaction_data = pd.read_csv(tranaction_path, low_memory =False)


contact_path= r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Payment\Contacts_Payment\Contacts.csv"
contact_data = pd.read_csv(contact_path, low_memory =False)
##print(contact_data.head(10))

JN= r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\JN_mapping.xlsx"
JN = pd.read_excel(JN)
##print(JN.head(10))

tranaction_data["live_response_code_flag"] = tranaction_data["live_response_code"].apply(lambda x: 'SUCCESS' if x=='SUCCESS' else 'Not_SUCCESS')
filtered_data = tranaction_data[(tranaction_data["payment_instrument"]!='COD') & (tranaction_data["merchant_id"]=='mp_flipkart') &(tranaction_data["hyp_flag"]=='HYP')].copy()
filtered_data["count_of_tx"] = pd.to_numeric(filtered_data["count_of_tx"], errors='coerce')
group_cols = ["transaction_yr", "transaction_mth","transaction_wk", "transaction_dt"]

overall_transaction_data = (filtered_data
                             .groupby(group_cols)["count_of_tx"]
                             .sum()
                             .reset_index()
                             .rename(columns={"count_of_tx": "overall_transaction_count"}))

successful_tranaction = (filtered_data[filtered_data["live_response_code_flag"]=="SUCCESS"]
                         .groupby(group_cols)["count_of_tx"].sum().reset_index()
                         .rename(columns={"count_of_tx":"successful_count"})) 

failed_data = (filtered_data[
               (filtered_data["live_response_code_flag"]=="Not_SUCCESS") &
                (filtered_data["transaction_status"]=="FAILED")]
               .groupby(group_cols)["count_of_tx"].sum().reset_index()
               .rename(columns={"count_of_tx": "failed_count"}))

adonc_data = (filtered_data[
              (filtered_data["live_response_code_flag"]=="Not_SUCCESS") &
              (filtered_data["transaction_status"]=="SUCCESS")]
             .groupby(group_cols)["count_of_tx"].sum().reset_index()
             .rename(columns={"count_of_tx": "adonc_count"})) 
order_confirmed_data = (filtered_data[
                        (filtered_data["live_response_code_flag"]=="Not_SUCCESS")&
                        (filtered_data["transaction_status"]=="SUCCESS")&
                        (filtered_data["merchant_status"]=="CAPTURED")]
                        .groupby(group_cols)["count_of_tx"].sum().reset_index()
                        .rename(columns={"count_of_tx": "order_confirmed_count"}))
transaction_failed_data = (filtered_data[
                        (filtered_data["live_response_code_flag"]=="Not_SUCCESS")&
                        (filtered_data["transaction_status"]=="SUCCESS")&
                        (filtered_data["merchant_status"]=="INVALIDATED")]
                        .groupby(group_cols)["count_of_tx"].sum().reset_index()
                        .rename(columns={"count_of_tx": "transaction_failed_count"}))



summary_frames = [
    overall_transaction_data.set_index("transaction_dt")[["overall_transaction_count"]].rename(columns={"overall_transaction_count": "Total_tranaction"}),
    successful_tranaction.set_index("transaction_dt")[["successful_count"]].rename(columns={"successful_count": "Successful_transaction"}),
    failed_data.set_index("transaction_dt")[["failed_count"]].rename(columns={"failed_count": "Failed_transaction"}),
    adonc_data.set_index("transaction_dt")[["adonc_count"]].rename(columns={"adonc_count": "adonc_transaction"}),
    order_confirmed_data.set_index("transaction_dt")[["order_confirmed_count"]].rename(columns={"order_confirmed_count": "order_confirmed_transaction"}),
    transaction_failed_data.set_index("transaction_dt")[["transaction_failed_count"]].rename(columns={"transaction_failed_count": "transaction_failed_transaction"})
       ]                     
summary_df = pd.concat(summary_frames, axis=1).fillna(0)
final_summary = summary_df.T
final_summary.columns = pd.to_datetime(final_summary.columns, errors='coerce').strftime('%d-%m-%Y') 
final_summary.reset_index(inplace=True)
final_summary.rename(columns={"Index": "Transaction_Data"}, inplace=True)
#print(final_summary)

## Overall_Transaction_summary##
total_by_transaction_source = (
        filtered_data.groupby(["transaction_source"])["count_of_tx"].sum().reset_index()
        .rename(columns={"count_of_tx":"Total_transaction"})
)

transaction_order = ['AndroidApp', 'iOSApp', 'MobileSite', 'ucweb', 'WEB']
filtered_data["transaction_source"] = pd.Categorical(filtered_data["transaction_source"], categories=transaction_order, ordered = True)
                                                    
total_by_transaction_source = (
        filtered_data.groupby(["transaction_dt","transaction_source"], observed= False)["count_of_tx"].sum().reset_index()
        .rename(columns={"count_of_tx":"Total_transaction_count"})
)
pivot_source = total_by_transaction_source.pivot(index = "transaction_source", 
                                                columns = "transaction_dt",
                                                values = "Total_transaction_count").fillna(0)
pivot_source.columns= pd.to_datetime(pivot_source.columns, errors='coerce').strftime('%d-%m-%Y')
pivot_source.reset_index(inplace=True)
#print(pivot_source)

## ADONC_transaction_summary###
filtered_data1 = tranaction_data[(tranaction_data["payment_instrument"]!='COD') & (tranaction_data["merchant_id"]=='mp_flipkart') & (tranaction_data["hyp_flag"]=="HYP") & (tranaction_data["adonc_flag"]=="Adonc")].copy()
filtered_data1["count_of_tx"] = pd.to_numeric(filtered_data1["count_of_tx"], errors='coerce')


Adonc_transaction_source = (
        filtered_data1.groupby(["transaction_source"])["count_of_tx"].sum().reset_index()
        .rename(columns={"count_of_tx":"Adonc_transaction"})
)

transaction_order = ['AndroidApp', 'iOSApp', 'MobileSite', 'ucweb', 'WEB']
filtered_data1["transaction_source"] = pd.Categorical(filtered_data1["transaction_source"], categories=transaction_order, ordered = True)
                                                    
Adonc_transaction_source = (
        filtered_data1.groupby(["transaction_dt","transaction_source"], observed= False)["count_of_tx"].sum().reset_index()
        .rename(columns={"count_of_tx":"Adonc_transaction"})
)
pivot_source1 = Adonc_transaction_source.pivot(index = "transaction_source", 
                                                columns = "transaction_dt",
                                                values = "Adonc_transaction").fillna(0)
pivot_source1.columns= pd.to_datetime(pivot_source1.columns, errors='coerce').strftime('%d-%m-%Y')
pivot_source1.reset_index(inplace=True)
##print(pivot_source1)

### Payment_instrument_overall##

filtered_data2 = tranaction_data[(tranaction_data["payment_instrument"]!='COD') & (tranaction_data["merchant_id"]=='mp_flipkart') &(tranaction_data["hyp_flag"]=='HYP')].copy()
filtered_data2["count_of_tx"] = pd.to_numeric(filtered_data2["count_of_tx"], errors='coerce')
filtered_data2["payment_instrument"] = filtered_data2["payment_instrument"].replace({"EGV":"EGV_wallet","EGV_WALLET":"EGV_wallet"})

Payment_instrument_source = (
        filtered_data2.groupby(["payment_instrument"])["count_of_tx"].sum().reset_index()
        .rename(columns={"count_of_tx":"Payment_instrument_transaction"})
)

transaction_order1 = ['CREDIT','DEBIT','FK_UPI','PHONEPE','UPI_COLLECT','UPI_INTENT']
filtered_data2["payment_instrument"] = pd.Categorical(filtered_data2["payment_instrument"], categories=transaction_order1, ordered = True)
                                                    
Payment_instrument_source = (
        filtered_data2.groupby(["transaction_dt","payment_instrument"], observed= False)["count_of_tx"].sum().reset_index()
        .rename(columns={"count_of_tx":"Payment_instrument_count"})
)
pivot_source2 = Payment_instrument_source.pivot(index = "payment_instrument", 
                                                columns = "transaction_dt",
                                                values = "Payment_instrument_count").fillna(0)
pivot_source2.columns= pd.to_datetime(pivot_source2.columns, errors='coerce').strftime('%d-%m-%Y')
pivot_source2.reset_index(inplace=True)
#print(pivot_source2)

### ADONC_instrument_overall##

filtered_data3 = tranaction_data[(tranaction_data["payment_instrument"]!='COD') & (tranaction_data["merchant_id"]=='mp_flipkart') & (tranaction_data["adonc_flag"]=='Adonc') & (tranaction_data["hyp_flag"]=='HYP')].copy()
filtered_data3["count_of_tx"] = pd.to_numeric(filtered_data3["count_of_tx"], errors='coerce')
filtered_data3["payment_instrument"] = filtered_data3["payment_instrument"].replace({"EGV":"EGV_wallet","EGV_WALLET":"EGV_wallet"})

Adonc_Payment_instrument_source = (
        filtered_data3.groupby(["payment_instrument"])["count_of_tx"].sum().reset_index()
        .rename(columns={"count_of_tx":"Adonc_Pay_instrument_transaction"})
)

transaction_order1 = ['CREDIT','DEBIT','FK_UPI','PHONEPE','UPI_COLLECT','UPI_INTENT']
filtered_data3["payment_instrument"] = pd.Categorical(filtered_data3["payment_instrument"], categories=transaction_order1, ordered = True)
                                                    
Adonc_Payment_instrument_source = (
        filtered_data3.groupby(["transaction_dt","payment_instrument"], observed= False)["count_of_tx"].sum().reset_index()
        .rename(columns={"count_of_tx":"Adonc_Pay_instrument_count"})
)
pivot_source3 = Adonc_Payment_instrument_source.pivot(index = "payment_instrument", 
                                                columns = "transaction_dt",
                                                values = "Adonc_Pay_instrument_count").fillna(0)
pivot_source3.columns= pd.to_datetime(pivot_source3.columns, errors='coerce').strftime('%d-%m-%Y')
pivot_source3.reset_index(inplace=True)

output_path =r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Hyperlocal_tranaction_output.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
   final_summary.to_excel(writer, sheet_name='final_summary', index = False) 
   pivot_source.to_excel(writer, sheet_name='Transaction_Source', index = False)
   pivot_source1.to_excel(writer, sheet_name = 'Adonc_Transaction_Source', index = False ) 
   pivot_source2.to_excel(writer, sheet_name = 'Payment_Instrument', index = False ) 
   pivot_source3.to_excel(writer, sheet_name = 'Adonc_Payment_Instrument', index = False ) 
print("Excel file successfully saved:", output_path)



Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Hyperlocal_tranaction_output.xlsx
